In [1]:
sample = 'mouse_skin'
save_dir = 'peak'

In [ ]:
import cv2
import glob
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import rotate
from skimage import transform as tf
from skimage.transform import warp
import scipy.io
import cv2
import matplotlib.pyplot as plt
import os

all_cell_mapping = pd.read_csv(sample + '/all_cell_mapping.csv', index_col=0)
all_cell_raman = pd.read_csv(sample + '/all_cell_raman.csv', index_col=0)
wave_number = scipy.io.loadmat(sample + '/wavenumbers.mat')['wavenumber'][0][413:1286].round(0).astype(np.int16)

In [ ]:
from scipy.integrate import simpson
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

def normalize_spectra(wavenum, intensity, min_peak, max_peak):
    amide_mask = (wavenum >= min_peak) & (wavenum <= max_peak)
    area = simpson(intensity[amide_mask], x=wavenum[amide_mask])
    # print(area)
    return intensity / area

# Function to generate distinct colors from the 'jet' colormap
def generate_tab20_extended_colors():
    colors = list(plt.cm.tab20(np.linspace(0, 1, 20)))  # Generate 20 colors from tab20
    additional_color = plt.cm.Set2(5)  # Select a color from another colormap, e.g., the second color in 'Set1'
    colors.append(additional_color)  # Append the additional color to the list
    return colors

# Generate 21 colors
extended_colors = generate_tab20_extended_colors()
# define each cell type a color
color_mapping = {cell_type: color for cell_type, color in zip(all_cell_mapping['cell_type'].unique(), extended_colors)}
print(color_mapping)

In [4]:
def plot_peak_diff(positive, negative, wave_number, label_set, cell_type, prefix):

    import numpy as np
    import pandas as pd
    import scanpy as sc

    data1 = pd.DataFrame(positive, columns=wave_number)
    data2 = pd.DataFrame(negative, columns=wave_number)
    # Calculate the Wilcoxon rank-sum test for each feature (column)
    p_values = []
    log2FC = []
    for column in data1.columns:
        stat, p_value = ranksums(data1[column], data2[column])
        p_values.append(p_value)

        mean_data1 = data1[column].mean()
        mean_data2 = data2[column].mean()
        log2FC.append(np.log2(mean_data1 / mean_data2))
        
    # Apply FDR correction
    _, p_values_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

    # Create a DataFrame with the original and corrected p-values
    results = pd.DataFrame({
            'Index': range(len(data1.columns)),
            'Peak': data1.columns,
            'p-value': p_values,
            'p-value_corrected': p_values_corrected,
            'log2FC': log2FC
        })

    # Filter features with FDR < 0.05
    significant_results = results[results['p-value_corrected'] <= 0.05]

    upregulated = significant_results[significant_results['log2FC'] > 0]
    downregulated = significant_results[significant_results['log2FC'] < 0]

    upregulated = upregulated.sort_values('log2FC', ascending=False)
    downregulated = downregulated.sort_values('log2FC', ascending=True)
    
    if len(upregulated) < 1 and len(downregulated) < 1:
        return None
    
    else:

        print('number of upregulated peaks:', len(upregulated))
        print('number of downregulated peaks:', len(downregulated))

        # plot the two average spectra
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8, 5))
        # set font size
        plt.rcParams.update({'font.size': 8})
        # set frnt style
        plt.rcParams.update({'font.family': 'Arial'})

        if prefix.startswith('Old'):
            plt.plot(wave_number, positive.mean(axis=0), label='Senescent' + ' (' + str(len(positive)) + ')', linewidth=1, alpha=0.5, color = '#EB382E')  # EB382E
            # add variance
            plt.fill_between(wave_number, positive.mean(axis=0) - positive.std(axis=0), positive.mean(axis=0) + positive.std(axis=0), alpha=0.2, linewidth=1, color = '#EB382E')

            plt.plot(wave_number, negative.mean(axis=0), label='Non-senescent' + ' (' + str(len(negative)) + ')', linewidth=1, alpha=0.5, color = '#91B9D6') # 91B9D6
            # add variance
            plt.fill_between(wave_number, negative.mean(axis=0) - negative.std(axis=0), negative.mean(axis=0) + negative.std(axis=0), alpha=0.2, linewidth=1, color = '#91B9D6')
            # add difference
            plt.plot(wave_number, positive.mean(axis=0) - negative.mean(axis=0), label='Differences',  linestyle='dotted', linewidth=1, color='gray')

        else:

            plt.plot(wave_number, positive.mean(axis=0), label='Old' + ' (' + str(len(positive)) + ')', linewidth=1, alpha=0.5, color = '#EB382E')  # EB382E
            # add variance
            plt.fill_between(wave_number, positive.mean(axis=0) - positive.std(axis=0), positive.mean(axis=0) + positive.std(axis=0), alpha=0.2, linewidth=1, color = '#EB382E')

            plt.plot(wave_number, negative.mean(axis=0), label='Young' + ' (' + str(len(negative)) + ')', linewidth=1, alpha=0.5, color = '#91B9D6') # 91B9D6
            # add variance
            plt.fill_between(wave_number, negative.mean(axis=0) - negative.std(axis=0), negative.mean(axis=0) + negative.std(axis=0), alpha=0.2, linewidth=1, color = '#91B9D6')
            # add difference
            plt.plot(wave_number, positive.mean(axis=0) - negative.mean(axis=0), label='Differences',  linestyle='dotted', linewidth=1, color='gray')

            
        plt.xlabel('Raman shift (cm$^{-1}$)')
        plt.ylabel('Intensity (a.u.)')
        plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
        plt.title(label_set + ' (' + cell_type + ')')
        # hide grid
        plt.grid(False)
        # tight layout
        # plt.tight_layout()|
        # hide y axis number
        plt.gca().axes.yaxis.set_ticklabels([])
        # hide scale
        # plt.gca().axes.get_yaxis().set_visible(False)
        # hide right and top axis
        plt.gca().spines['right'].set_visible(False)
        plt.gca().spines['top'].set_visible(False)
        plt.tight_layout()
        # # save 

        if '/' in cell_type:
            cell_type = cell_type.replace('/', '_')


        save_path = 'figures' + '/' + sample + '/' + save_dir
        if not os.path.exists(save_path):
            os.makedirs(save_path)

        plt.savefig(f'{save_path}/{prefix}_{label_set}_{cell_type}.pdf')

        plt.show()


In [ ]:
from scipy import stats

label_sets = [ 'old']
tiltle_sets = ['old']

for label_set,tiletle_set in zip(label_sets, tiltle_sets):

    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O'] # select only old cells

    if len(selected_cell_mapping) < 1:
        continue

    selected_cell_raman = all_cell_raman.loc[selected_cell_mapping.index]

    labels = selected_cell_mapping['p21+'].values * 1
    features = selected_cell_raman.values.astype(np.float32)
    
    new_features = []
    for i in range(features.shape[0]):
        new_features.append(normalize_spectra(wave_number, features[i], 1630, 1700))
    features = np.stack(new_features, axis=0)
    
    positive = features[labels == 1]
    negative = features[labels == 0]
    
    if len(positive) < 1 and len(negative) < 1:
        continue

    plot_peak_diff(positive, negative, wave_number, label_set, 'global', 'All_SvsNS')


# cell_type
for label_set,tiletle_set in zip(label_sets, tiltle_sets):

    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O'] # select only old cells
    cell_types = set(selected_cell_mapping['cell_type'])

    for cell_type in cell_types: 

        selected_cell_type_mapping = selected_cell_mapping[selected_cell_mapping['cell_type'] == cell_type]

        if len(selected_cell_mapping) < 1:
            continue

        selected_cell_raman = all_cell_raman.loc[selected_cell_type_mapping.index]

        labels = selected_cell_type_mapping['p21+'].values * 1
        features = selected_cell_raman.values.astype(np.float32)
        
        new_features = []
        for i in range(features.shape[0]):
            new_features.append(normalize_spectra(wave_number, features[i], 1630, 1700))
        features = np.stack(new_features, axis=0)
        
        positive = features[labels == 1]
        negative = features[labels == 0]
        
        if len(positive) < 1 and len(negative) < 1:
            continue

        plot_peak_diff(positive, negative, wave_number, label_set, cell_type, 'All_SvsNS')
